# Notebook 04 — Reranker Ablation

**Goal:** Compare hybrid retrieval with and without cross-encoder reranking.
The cross-encoder sees the full (query, chunk) pair — more accurate but slower.
Quantify the quality vs. latency trade-off.

**Output:** `figures/04_reranker_ablation.png`, latency + quality comparison table

In [ ]:
import sys, json, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('dark_background')

sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd()))
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# ── Recall@1 with and without reranker ──────────────────────────────────
from src.retriever import Retriever
from src.eval import load_test_queries, recall_at_k, reciprocal_rank
import time

queries = [q for q in load_test_queries() if q.get('query_type') in ('in_corpus', 'borderline')]

for label, use_reranker in [('No reranker', False), ('With reranker', True)]:
    r_no = Retriever(use_hybrid=True, use_reranker=use_reranker, top_k=4)
    r1_scores, mrr_scores, times = [], [], []
    for q in queries:
        t0 = time.time()
        chunks = r_no.retrieve(q['query'], top_k=4)
        elapsed = time.time() - t0
        r1_scores.append(recall_at_k(chunks, q['expected_source_doc'], q['expected_source_pages'], k=1))
        mrr_scores.append(reciprocal_rank(chunks, q['expected_source_doc'], q['expected_source_pages']))
        times.append(elapsed)
    print(f'{label:20s}  Recall@1={sum(r1_scores)/len(r1_scores):.3f}  MRR={sum(mrr_scores)/len(mrr_scores):.3f}  avg_latency={sum(times)/len(times)*1000:.0f}ms')

In [ ]:
# ── Latency distribution: no reranker vs reranker ───────────────────────
import numpy as np

# (Re-run the loop above storing times_no_rerank and times_rerank separately)
# Then plot:
# fig, ax = plt.subplots(figsize=(8, 4))
# ax.hist(times_no_rerank, bins=15, alpha=0.7, label='No reranker', color='#4f8ef7')
# ax.hist(times_rerank, bins=15, alpha=0.7, label='With reranker', color='#7dd3a8')
# ax.set_xlabel('Latency (s)')
# ax.set_ylabel('Queries')
# ax.set_title('Retrieval Latency: Reranker On vs Off')
# ax.legend()
# plt.tight_layout()
# plt.savefig('figures/04_reranker_ablation.png', dpi=150, bbox_inches='tight')
# plt.show()
print('Uncomment and run once times_no_rerank / times_rerank are collected above')

## Reranker Ablation Results

| Config | Recall@1 | MRR | Avg Latency |
|--------|----------|-----|-------------|
| Hybrid (no reranker) | *(fill in)* | *(fill in)* | *(fill in)* ms |
| Hybrid + cross-encoder rerank | *(fill in)* | *(fill in)* | *(fill in)* ms |

**Key finding:** *(e.g. 'Reranker improves Recall@1 by +X%, MRR by +Y, at a cost of +Z ms median latency. Worth the trade-off for the deployed system.')*

**Final configuration:** Hybrid (dense + BM25 RRF) + cross-encoder rerank, top_k=4, chunk_size=500

**Next step:** Run `uvicorn api.main:app --reload` and test the API locally before deployment.